# Coupled neutron-proton scattering with `lax.models`

You do **not** need the Descouvemont paper to follow this notebook. The goal here is simply to solve a two-channel scattering problem where an incoming neutron-proton state can mix an `S` wave and a `D` wave.

We will:

1. load one published benchmark grid from JSON,
2. inspect the public Reid soft-core interaction helpers in `lax.models`,
3. compile a coupled-channel solver, and
4. compare three easy-to-read observables to the checked-in reference values:
   - two eigenphases, which summarize the two coupled scattering modes,
   - and `|S12|`, the off-diagonal coupling strength.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

import lax as lm


def benchmark_data_dir() -> Path:
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    search_roots.append(Path(lm.__file__).resolve().parents[2])
    for root in search_roots:
        candidate = root / 'tests' / 'benchmarks' / 'data'
        if candidate.is_dir():
            return candidate
    msg = 'Could not locate tests/benchmarks/data from the current notebook environment.'
    raise FileNotFoundError(msg)

fixture = json.loads((benchmark_data_dir() / 'descouvemont_np_j1.json').read_text())
reference = fixture['references'][0]

energies = np.asarray(reference['energies'], dtype=np.float64)
channels = lm.models.reid_np_j1_channels()
mesh = lm.MeshSpec('legendre', 'x', n=int(reference['n_basis']), scale=float(reference['scale']))

{
    'energies_mev': energies.tolist(),
    'channels': [
        {'index': index, 'l': channel.l, 'threshold_mev': channel.threshold}
        for index, channel in enumerate(channels)
    ],
}


## The public interaction helpers

`lax.models.reid_soft_core_triplet_components(...)` returns the three radial pieces of the Reid soft-core triplet interaction:

- a central term,
- a tensor term, which is what mixes the `S` and `D` waves,
- and a spin-orbit term.

The public helper `lax.models.reid_np_j1_potential(...)` combines those pieces into the `2 × 2` local potential that `lax.assemble_local(...)` expects.


In [ ]:
sample_radii = np.asarray([0.5, 1.0, 2.0, 4.0, 6.0], dtype=np.float64)
central, tensor, spin_orbit = [
    np.asarray(values)
    for values in lm.models.reid_soft_core_triplet_components(sample_radii)
]

[
    {
        'r_fm': float(radius),
        'central_mev': float(v_c),
        'tensor_mev': float(v_t),
        'spin_orbit_mev': float(v_ls),
    }
    for radius, v_c, v_t, v_ls in zip(sample_radii, central, tensor, spin_orbit, strict=True)
]


## Solve the coupled scattering problem

We now compile a spectral solver on the benchmark energy grid, build the coupled local potential from the public model helper, and extract the observable quantities from the `S` matrix.


In [ ]:
solver = lm.compile(
    mesh=mesh,
    channels=channels,
    operators=('T+L', '1/r^2'),
    solvers=('spectrum', 'smatrix'),
    energies=energies,
    method='eigh',
)
potential = lm.assemble_local(
    solver.mesh,
    lm.models.reid_np_j1_potential,
    n_channels=len(channels),
)
spectrum = solver.spectrum(potential)
smatrices = np.asarray(solver.smatrix(spectrum))

rows = []
for energy, smatrix, ref_phase_11, ref_phase_22, ref_eta_12 in zip(
    energies,
    smatrices,
    reference['phase_11'],
    reference['phase_22'],
    reference['eta_12'],
    strict=True,
):
    params = lm.spectral.coupled_channel_parameters_from_S(smatrix)
    rows.append(
        {
            'energy_mev': float(energy),
            'computed_phase_11': float(np.asarray(params.phase_2)),
            'reference_phase_11': float(ref_phase_11),
            'computed_phase_22': float(np.asarray(params.phase_1)),
            'reference_phase_22': float(ref_phase_22),
            'computed_abs_s12': float(abs(smatrix[0, 1])),
            'reference_abs_s12': float(ref_eta_12),
        }
    )

rows


## How to adapt this notebook

This workflow is intentionally built from reusable pieces:

- swap `mesh` to study convergence,
- replace `energies` with a denser scan,
- or replace `lm.models.reid_np_j1_potential` with another `2 × 2` coupled interaction callback.

The notebook structure stays the same: define channels, assemble a potential, compile the solver, then interpret the `S` matrix in whatever basis is most useful for your application.
